In [2]:
import pandas as pd

from ultralytics import YOLO, models

data = "VisDrone"
# data = "DOTA"


models = []
data = []

if data == "DOTA":
    models = ["yolov12n-DOTA"]
    data = ["DOTAv1.5", "DOTAv1.5-SMALL", "DOTAv1.5-MEDIUM", "DOTAv1.5-LARGE"]
elif data == "VisDrone":
    models = ["yolov12n-VisDrone"]
    data = ["VisDrones-SMALL", "VisDrones-MEDIUM", "VisDrones-LARGE"]

df_dota_object_stats = []
for _model in models:
    model = YOLO(f"resultats/{_model}-200.pt")

    for data in data:
        results = model.val(
            data=f"/Users/osias/Documents/PHD/CODE/ultralytics-git/code/ultralytics/cfg/datasets/{data}.yaml",
            plots=True,
        )
        line = {
            "DataSet": data,
            "Model": _model,
            "P": results.results_dict["metrics/precision(B)"],
            "R": results.results_dict["metrics/recall(B)"],
            "mAP50": results.results_dict["metrics/mAP50(B)"],
            "mAP50-95": results.results_dict["metrics/mAP50-95(B)"],
            "preprocess": results.speed["preprocess"],
            "inference": results.speed["inference"],
            "postprocess": results.speed["postprocess"],
            "loss": results.speed["loss"],
        }
        df_dota_object_stats.append(line)

df_dota_object_stats = pd.DataFrame(df_dota_object_stats)
df_dota_object_stats.to_csv(f"resultats/results_{data}.csv", index=False, sep=";")

Ultralytics 8.4.33 🚀 Python-3.9.18 torch-2.7.1 CPU (Apple M3 Pro)
YOLO12 summary (fused): 159 layers, 2,558,678 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 322.8±85.1 MB/s, size: 128.8 KB)
val: Scanning /Users/osias/Documents/PHD/CODE/datasets/VisDrone-SMALL/val/labels.cache... 524 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 524/524 28.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 33/33 1.8s/it 59.2s1.7ss
                   all        524      26588      0.186      0.267      0.124     0.0516
            pedestrian        494       7746      0.321      0.365      0.245     0.0925
                people        462       4631      0.373      0.287      0.241     0.0882
               bicycle        313       1053      0.123       0.11     0.0455     0.0161
                   car        442       6911      0.243      0.652      0.222      0.108
                   van

In [7]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image


# --- Self-attention simple (mock pour test) ---
def self_attention(x, heads=8):
    # x: (B, H, W, C)
    B, H, W, C = x.shape
    x = x.view(B, H * W, C)

    attn = torch.softmax(torch.matmul(x, x.transpose(-1, -2)) / (C**0.5), dim=-1)
    out = torch.matmul(attn, x)

    return out.view(B, H, W, C)


# --- Ton module ---
class CrossAreaAttention(nn.Module):
    def __init__(self, dim, l=4, heads=8):
        super().__init__()
        self.l = l
        self.heads = heads
        self.alpha = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        B, H, W, C = x.shape

        assert H % self.l == 0 and W % self.l == 0

        xv = x.view(B * self.l, H // self.l, W, C)
        xv = self_attention(xv, self.heads)
        out_V = xv.view(B, H, W, C)

        xh = x.view(B * self.l, H, W // self.l, C)
        xh = self_attention(xh, self.heads)
        out_H = xh.view(B, H, W, C)

        alpha = torch.sigmoid(self.alpha)
        out = alpha * out_V + (1 - alpha) * out_H

        return out


# --- Charger une image ---
image = Image.open("/Users/osias/Documents/PHD/ultralytics-osias/ultralytics/assets/zidane.jpg").convert("RGB")

transform = T.Compose([T.Resize((128, 128)), T.ToTensor()])

img_tensor = transform(image)  # (C, H, W)
img_tensor = img_tensor.permute(1, 2, 0)  # (H, W, C)
img_tensor = img_tensor.unsqueeze(0)  # (1, H, W, C)

# --- Appliquer module ---
model = CrossAreaAttention(dim=3, l=8)
output = model(img_tensor)

# --- Affichage ---
out_img = output.squeeze(0).permute(2, 0, 1).detach()

plt.imshow(out_img.permute(1, 2, 0))
plt.title("Output Image")
plt.axis("off")
plt.show()